# 02: Query and visualize the weather data

This notebook runs equivalent Athena queries against raw CSV and curated Parquet, compares bytes scanned, charts five years of weather observations, and saves an analysis table.

Expected time: 25 minutes. Run `01_build_pipeline.ipynb` first.

## Load the project configuration and verify identity

The generated configuration contains resource names, not credentials. STS is called again so a profile or role change cannot silently query the wrong account.

In [ ]:
from pathlib import Path
import json
import os
import time

import boto3
import matplotlib.pyplot as plt
import pandas as pd

config_path = Path("project_config.json")
if not config_path.exists():
    raise FileNotFoundError("Run 01_build_pipeline.ipynb before this notebook")
# Reuse the AWS resource names saved by Notebook 1.
config = json.loads(config_path.read_text())

AWS_PROFILE = os.getenv("AWS_PROFILE") or None
session_args = {"region_name": config["region"]}
if AWS_PROFILE:
    session_args["profile_name"] = AWS_PROFILE
session = boto3.Session(**session_args)

identity = session.client("sts").get_caller_identity()
if identity["Account"] != config["account_id"]:
    raise RuntimeError(
        f"Current account {identity['Account']} does not match project account {config['account_id']}"
    )

print("AWS project connection")
print(f"  Account ID:        {identity['Account']}")
print(f"  Principal ARN:     {identity['Arn']}")
print(f"  Region:            {config['region']}")
print(f"  Glue database:     {config['database']}")
print(f"  Athena workgroup:  {config['workgroup']}")

## Run Athena through boto3

Athena is asynchronous. The helper starts a query, polls its state, raises the returned failure reason, collects paginated rows, and keeps the execution statistics.

In [ ]:
athena = session.client("athena")

def run_athena(sql, timeout_seconds=120):
    # Athena runs asynchronously: start it, then poll the execution ID.
    start = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": config["database"]},
        WorkGroup=config["workgroup"],
    )
    execution_id = start["QueryExecutionId"]
    deadline = time.monotonic() + timeout_seconds

    while True:
        execution = athena.get_query_execution(QueryExecutionId=execution_id)["QueryExecution"]
        state = execution["Status"]["State"]
        if state == "SUCCEEDED":
            break
        if state in {"FAILED", "CANCELLED"}:
            reason = execution["Status"].get("StateChangeReason", "No reason returned")
            raise RuntimeError(f"Athena query {state.lower()}: {reason}")
        if time.monotonic() >= deadline:
            athena.stop_query_execution(QueryExecutionId=execution_id)
            raise TimeoutError(f"Athena query exceeded {timeout_seconds} seconds")
        time.sleep(1)

    rows = []
    columns = []
    # Athena may return results in multiple pages.
    paginator = athena.get_paginator("get_query_results")
    for page in paginator.paginate(QueryExecutionId=execution_id):
        if not columns:
            columns = [
                column["Name"]
                for column in page["ResultSet"]["ResultSetMetadata"]["ColumnInfo"]
            ]
        for row in page["ResultSet"]["Rows"]:
            values = [cell.get("VarCharValue") for cell in row["Data"]]
            rows.append(values)
    if rows and rows[0] == columns:
        rows = rows[1:]

    statistics = execution.get("Statistics", {})
    return {
        "execution_id": execution_id,
        "data": pd.DataFrame(rows, columns=columns),
        "scanned_bytes": int(statistics.get("DataScannedInBytes", 0)),
        "engine_ms": int(statistics.get("EngineExecutionTimeInMillis", 0)),
    }

## Compare raw CSV with curated Parquet

Both statements calculate annual average temperature for the first configured station. Their output should match. The execution statistics show how many bytes Athena read from each storage format.

In [ ]:
comparison_station = config["stations"][0]
comparison_station_id = comparison_station["station_id"]
comparison_city_key = comparison_station["city_key"]
print(f"Comparing storage formats for {comparison_station['city']}")

# Ask raw CSV and Parquet for the same metric.
raw_sql = f"""
SELECT year,
       ROUND(AVG(TRY_CAST(NULLIF(TRIM(temp), '9999.9') AS DOUBLE)), 2) AS avg_temp_f
FROM gsod_raw
WHERE station_id = '{comparison_station_id}'
GROUP BY year
ORDER BY year
"""

parquet_sql = f"""
SELECT year,
       ROUND(AVG(temp_f), 2) AS avg_temp_f
FROM weather_curated
WHERE city_key = '{comparison_city_key}'
GROUP BY year
ORDER BY year
"""

raw_result = run_athena(raw_sql)
parquet_result = run_athena(parquet_sql)

query_results = pd.concat(
    {
        "Raw CSV": raw_result["data"].reset_index(drop=True),
        "Partitioned Parquet": parquet_result["data"].reset_index(drop=True),
    },
    names=["format"],
)
query_results

In [ ]:
scan_comparison = pd.DataFrame([
    {
        "format": "Raw CSV",
        "bytes_scanned": raw_result["scanned_bytes"],
        "engine_ms": raw_result["engine_ms"],
    },
    {
        "format": "Partitioned Parquet",
        "bytes_scanned": parquet_result["scanned_bytes"],
        "engine_ms": parquet_result["engine_ms"],
    },
])
scan_comparison["kilobytes_scanned"] = (scan_comparison["bytes_scanned"] / 1024).round(1)
raw_values = raw_result["data"].astype(str).reset_index(drop=True)
parquet_values = parquet_result["data"].astype(str).reset_index(drop=True)
# Different storage formats must return the same values.
assert raw_values.equals(parquet_values), "CSV and Parquet query results do not match"
scan_comparison

In [ ]:
# Matplotlib separates the whole figure from its plotting area (ax).
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(scan_comparison["format"], scan_comparison["kilobytes_scanned"], color=["#64748b", "#2563eb"])
ax.set(title="Athena data scanned for the same result", ylabel="Kilobytes scanned")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Analyze the three cities

The next statement reads the curated table and returns one row per city and year. The calculations stay in Athena; pandas converts the result types and prepares the charts.

In [ ]:
city_keys_sql = ", ".join(
    f"'{station['city_key']}'"
    for station in config["stations"]
)
city_summary_sql = f"""
SELECT
    city,
    year,
    ROUND(AVG(temp_f), 2) AS avg_temp_f,
    ROUND(AVG(max_temp_f), 2) AS avg_daily_max_f,
    SUM(CASE WHEN is_hot_day THEN 1 ELSE 0 END) AS hot_days,
    SUM(CASE WHEN is_freezing_day THEN 1 ELSE 0 END) AS freezing_days,
    ROUND(SUM(precipitation_in), 2) AS precipitation_in,
    COUNT(*) AS observation_days
FROM weather_curated
WHERE city_key IN ({city_keys_sql})
GROUP BY city, year
ORDER BY city, year
"""

summary_result = run_athena(city_summary_sql)
city_year = summary_result["data"].copy()
numeric_columns = [
    "year", "avg_temp_f", "avg_daily_max_f", "hot_days",
    "freezing_days", "precipitation_in", "observation_days",
]
city_year[numeric_columns] = city_year[numeric_columns].apply(pd.to_numeric)
city_year

In [ ]:
Path("outputs").mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for city, group in city_year.groupby("city"):
    ordered = group.sort_values("year")
    axes[0].plot(ordered["year"], ordered["avg_temp_f"], marker="o", label=city)
    axes[1].plot(ordered["year"], ordered["hot_days"], marker="o", label=city)

axes[0].set(title="Annual average temperature", xlabel="Year", ylabel="Temperature (F)")
axes[1].set(title="Days with a high of at least 90 F", xlabel="Year", ylabel="Days")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(title="City")
    ax.set_xticks(config["years"])

plt.tight_layout()
chart_path = Path("outputs/weather_trends.png")
fig.savefig(chart_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved {chart_path}")

## Record the findings

Five years from one airport station per city is too small a sample for a climate claim. The code below reports observed differences without assigning a cause.

In [ ]:
findings = []
for city, group in city_year.groupby("city"):
    ordered = group.sort_values("year")
    first = ordered.iloc[0]
    last = ordered.iloc[-1]
    finding = {
        "city": city,
        "first_year": int(first["year"]),
        "last_year": int(last["year"]),
        "first_avg_temp_f": float(first["avg_temp_f"]),
        "last_avg_temp_f": float(last["avg_temp_f"]),
        "difference_f": round(float(last["avg_temp_f"] - first["avg_temp_f"]), 2),
        "hottest_year_in_sample": int(group.loc[group["avg_temp_f"].idxmax(), "year"]),
        "most_hot_days_in_sample": int(group["hot_days"].max()),
    }
    findings.append(finding)

findings_frame = pd.DataFrame(findings)
findings_frame

In [ ]:
# Save calculated outputs locally and in S3 for Notebook 3.
summary_path = Path("outputs/city_year_summary.csv")
findings_path = Path("outputs/calculated_findings.json")
city_year.to_csv(summary_path, index=False)
findings_path.write_text(json.dumps(findings, indent=2))

s3 = session.client("s3")
s3.upload_file(str(summary_path), config["bucket"], "analytics/city_year_summary.csv")
s3.upload_file(str(findings_path), config["bucket"], "analytics/calculated_findings.json")

print(f"Saved {summary_path} and {findings_path}")
print(f"Athena scanned {summary_result['scanned_bytes']:,} bytes for the city summary")

## Optional report

Open `03_bedrock_report.ipynb` to send `outputs/calculated_findings.json` to a Bedrock text model. The model receives calculated findings rather than raw rows.

## Cleanup

The next cell is disabled. Change `DELETE_RESOURCES` to `True` only when you are ready to delete the Glue tables and database, Athena workgroup, every object in the project bucket, and the bucket itself. This cannot be undone.

In [ ]:
DELETE_RESOURCES = False  # Safety switch: cleanup runs only when set to True.

if DELETE_RESOURCES:
    current_identity = session.client("sts").get_caller_identity()
    if current_identity["Account"] != config["account_id"]:
        raise RuntimeError("Refusing cleanup because the active account changed")

    glue = session.client("glue")
    for table_name in ("gsod_raw", "weather_curated"):
        try:
            glue.delete_table(DatabaseName=config["database"], Name=table_name)
        except glue.exceptions.EntityNotFoundException:
            pass
    try:
        glue.delete_database(Name=config["database"])
    except glue.exceptions.EntityNotFoundException:
        pass

    athena.delete_work_group(
        WorkGroup=config["workgroup"],
        RecursiveDeleteOption=True,
    )

    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=config["bucket"]):
        objects = [{"Key": item["Key"]} for item in page.get("Contents", [])]
        if objects:
            s3.delete_objects(Bucket=config["bucket"], Delete={"Objects": objects})
    s3.delete_bucket(Bucket=config["bucket"])
    config_path.unlink(missing_ok=True)
    print("Deleted the project AWS resources and project_config.json")
else:
    print("Cleanup skipped. Set DELETE_RESOURCES = True when you are ready.")